In [23]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import os
import pickle
import warnings
from datetime import datetime

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Create directory structure if it doesn't exist
base_dir = r"C:\Users\Public\Placement\Student-Performace-Prediction"
data_dir = os.path.join(base_dir, "Data")
models_dir = os.path.join(base_dir, "Models")
plots_dir = os.path.join(base_dir, "Plots")

for directory in [data_dir, models_dir, plots_dir]:
    if not os.path.exists(directory):
        os.makedirs(directory)

# Set random seed for reproducibility
np.random.seed(42)

# Function to generate realistic synthetic data
def generate_synthetic_data(n_samples=500):
    """
    Generate synthetic student performance data with realistic relationships
    between study habits, attendance, and performance scores.
    """
    print("Generating synthetic student data...")
    
    # Generate independent features with realistic distributions
    study_hours = np.random.normal(20, 7, n_samples).clip(5, 40)
    attendance = np.random.normal(85, 10, n_samples).clip(40, 100)
    previous_grades = np.random.normal(75, 15, n_samples).clip(30, 100)
    sleep_hours = np.random.normal(7, 1.5, n_samples).clip(4, 10)
    
    # Create extracurricular participation (30% yes, 70% no)
    extracurricular = np.random.choice(['Yes', 'No'], size=n_samples, p=[0.3, 0.7])
    
    # Create clear relationships for performance score
    # Base formula: 40% from previous grades, 25% from study habits, 
    # 15% from attendance, 10% from sleep, 10% from extracurricular
    
    # Start with previous grades influence
    performance = 0.4 * previous_grades
    
    # Add study hours influence (diminishing returns after 25 hours)
    study_effect = np.where(
        study_hours <= 25, 
        study_hours * 0.8, 
        25 * 0.8 + (study_hours - 25) * 0.3
    )
    performance += 0.25 * (study_effect * 100 / 32)  # Normalize to 0-100 scale
    
    # Add attendance influence
    performance += 0.15 * attendance
    
    # Add sleep influence (optimal around 7-8 hours)
    sleep_effect = 100 - 15 * np.abs(sleep_hours - 7.5)**1.5
    performance += 0.1 * sleep_effect
    
    # Add extracurricular influence
    extracurricular_effect = np.where(extracurricular == 'Yes', 100, 85)
    performance += 0.1 * extracurricular_effect
    
    # Add some realistic noise to make prediction challenging
    performance += np.random.normal(0, 7, n_samples)
    
    # Ensure performance is within 0-100 range
    performance = performance.clip(30, 100)
    
    # Create DataFrame
    data = pd.DataFrame({
        'Student_ID': range(1, n_samples + 1),
        'Study Hours per Week': np.round(study_hours, 1),
        'Attendance Rate': np.round(attendance, 1),
        'Previous Grades': np.round(previous_grades, 1),
        'Sleep Hours per Night': np.round(sleep_hours, 1),
        'Extra Curricular Activities': extracurricular,
        'Performance Score': np.round(performance, 1)
    })
    
    return data

# Main class for Student Performance Prediction
class StudentPerformancePredictor:
    def __init__(self, data_path=None, use_synthetic=True, n_samples=500):
        """
        Initialize the predictor with data loading or generation
        """
        self.creation_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.user = "ShivamKashyap"
        
        print("\n" + "="*70)
        print("                STUDENT PERFORMANCE PREDICTION SYSTEM")
        print("="*70)
        print(f"Created by: {self.user} | Date: {self.creation_date}\n")
        
        # Load or generate data
        if use_synthetic or data_path is None:
            self.data = generate_synthetic_data(n_samples)
            self.data_source = "synthetic"
            
            # Save synthetic data
            self.data_path = os.path.join(data_dir, "synthetic_student_data.csv")
            self.data.to_csv(self.data_path, index=False)
            print(f"Synthetic data saved to: {self.data_path}")
        else:
            # Load actual data if path provided
            self.data_path = data_path
            self.data = pd.read_csv(data_path)
            self.data_source = "actual"
            print(f"Data loaded from: {self.data_path}")
        
        # Set the target column
        self.target_column = 'Performance Score'
        self.id_column = 'Student_ID'
        
        # Display basic information
        print(f"\nDataset shape: {self.data.shape}")
        print(f"Target variable: {self.target_column}")
        print("\nSample data:")
        print(self.data.head())

    def explore_data(self):
        """
        Perform exploratory data analysis with visualizations
        """
        print("\n" + "="*70)
        print("                     DATA EXPLORATION")
        print("="*70)
        
        # Basic statistics
        print("\nBasic Statistics:")
        print(self.data.describe().round(2))
        
        # Missing values check
        missing = self.data.isnull().sum()
        print("\nMissing Values:")
        print(missing)
        
        # Distribution of target variable
        plt.figure(figsize=(10, 6))
        sns.histplot(self.data[self.target_column], kde=True)
        plt.title('Distribution of Performance Scores')
        plt.xlabel('Performance Score')
        plt.ylabel('Frequency')
        plt.savefig(os.path.join(plots_dir, "performance_distribution.png"))
        plt.close()
        
        # Correlation analysis
        numeric_data = self.data.select_dtypes(include=['float64', 'int64'])
        correlation = numeric_data.corr()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
        plt.title('Correlation Between Variables')
        plt.savefig(os.path.join(plots_dir, "correlation_heatmap.png"))
        plt.close()
        
        # Relationship between each feature and target
        for feature in numeric_data.columns:
            if feature != self.target_column and feature != self.id_column:
                plt.figure(figsize=(10, 6))
                sns.scatterplot(x=feature, y=self.target_column, data=self.data)
                plt.title(f'Relationship between {feature} and {self.target_column}')
                plt.xlabel(feature)
                plt.ylabel(self.target_column)
                plt.savefig(os.path.join(plots_dir, f"{feature}_scatter.png"))
                plt.close()
        
        # Categorical feature analysis
        plt.figure(figsize=(10, 6))
        sns.boxplot(x='Extra Curricular Activities', y=self.target_column, data=self.data)
        plt.title('Performance by Extracurricular Participation')
        plt.savefig(os.path.join(plots_dir, "extracurricular_boxplot.png"))
        plt.close()
        
        print(f"\nExploratory visualizations saved to: {plots_dir}")
        
        # Show key insights
        print("\nKey Insights:")
        print(f"- Average Performance Score: {self.data[self.target_column].mean():.2f}")
        print(f"- Most important correlations with Performance Score:")
        for feature, corr in correlation[self.target_column].sort_values(ascending=False).items():
            if feature != self.target_column:
                print(f"  * {feature}: {corr:.3f}")

    def prepare_data(self):
        """
        Prepare data for modeling by creating features and splitting dataset
        """
        print("\n" + "="*70)
        print("                      DATA PREPARATION")
        print("="*70)
        
        # Convert categorical to numeric
        self.data['Extra Curricular Activities'] = self.data['Extra Curricular Activities'].map({'Yes': 1, 'No': 0})
        
        # Feature engineering
        print("\nPerforming feature engineering...")
        # Study efficiency (grades per hour of study)
        self.data['Study_Efficiency'] = self.data['Previous Grades'] / self.data['Study Hours per Week']
        
        # Interaction terms
        self.data['Study_Attendance'] = self.data['Study Hours per Week'] * self.data['Attendance Rate'] / 100
        
        # Optimal sleep indicator (how close to optimal 8 hours)
        self.data['Sleep_Optimality'] = 1 - abs(self.data['Sleep Hours per Night'] - 8) / 8
        
        # Define features and target
        self.feature_columns = [col for col in self.data.columns 
                                if col != self.target_column 
                                and col != self.id_column]
        
        print(f"Features after engineering: {len(self.feature_columns)}")
        print(self.feature_columns)
        
        # Split data
        X = self.data[self.feature_columns]
        y = self.data[self.target_column]
        
        # Create train/validation/test split (70/15/15)
        X_temp, self.X_test, y_temp, self.y_test = train_test_split(
            X, y, test_size=0.15, random_state=42)
        
        self.X_train, self.X_val, self.y_train, self.y_val = train_test_split(
            X_temp, y_temp, test_size=0.1765, random_state=42)  # ~15% of total
        
        print(f"\nTraining set: {self.X_train.shape[0]} samples")
        print(f"Validation set: {self.X_val.shape[0]} samples")
        print(f"Test set: {self.X_test.shape[0]} samples")
        
        # Standardize numerical features
        self.scaler = StandardScaler()
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_val_scaled = self.scaler.transform(self.X_val)
        self.X_test_scaled = self.scaler.transform(self.X_test)

    def train_models(self):
        """
        Train and compare multiple models
        """
        print("\n" + "="*70)
        print("                      MODEL TRAINING")
        print("="*70)
        
        # Define models to try
        models = {
            'Linear Regression': LinearRegression(),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
        }
        
        # Train and evaluate each model
        self.model_results = {}
        
        for name, model in models.items():
            print(f"\nTraining {name}...")
            
            # Train the model
            model.fit(self.X_train_scaled, self.y_train)
            
            # Cross-validation
            cv_scores = cross_val_score(
                model, self.X_train_scaled, self.y_train, 
                cv=5, scoring='r2'
            )
            
            # Validation performance
            val_pred = model.predict(self.X_val_scaled)
            val_r2 = r2_score(self.y_val, val_pred)
            val_rmse = np.sqrt(mean_squared_error(self.y_val, val_pred))
            val_mae = mean_absolute_error(self.y_val, val_pred)
            
            # Store results
            self.model_results[name] = {
                'model': model,
                'cv_r2_mean': np.mean(cv_scores),
                'cv_r2_std': np.std(cv_scores),
                'val_r2': val_r2,
                'val_rmse': val_rmse,
                'val_mae': val_mae
            }
            
            # Print results
            print(f"  Cross-Validation R² Score: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
            print(f"  Validation R² Score: {val_r2:.4f}")
            print(f"  Validation RMSE: {val_rmse:.2f}")
            print(f"  Validation MAE: {val_mae:.2f}")
        
        # Identify best model based on validation R²
        self.best_model_name = max(self.model_results, 
                                  key=lambda k: self.model_results[k]['val_r2'])
        
        self.best_model = self.model_results[self.best_model_name]['model']
        print(f"\nBest model: {self.best_model_name} with validation R² of "
              f"{self.model_results[self.best_model_name]['val_r2']:.4f}")

    def evaluate_best_model(self):
        """
        Evaluate the best model on the test set and analyze its performance
        """
        print("\n" + "="*70)
        print("                    MODEL EVALUATION")
        print("="*70)
        
        # Test set predictions
        test_pred = self.best_model.predict(self.X_test_scaled)
        
        # Calculate metrics
        test_r2 = r2_score(self.y_test, test_pred)
        test_rmse = np.sqrt(mean_squared_error(self.y_test, test_pred))
        test_mae = mean_absolute_error(self.y_test, test_pred)
        
        # Store test metrics
        self.test_metrics = {
            'r2': test_r2,
            'rmse': test_rmse,
            'mae': test_mae
        }
        
        # Print test performance
        print(f"\nTest Performance for {self.best_model_name}:")
        print(f"  R² Score: {test_r2:.4f}")
        print(f"  Root Mean Squared Error: {test_rmse:.2f}")
        print(f"  Mean Absolute Error: {test_mae:.2f}")
        
        # Visualize actual vs predicted
        plt.figure(figsize=(10, 6))
        plt.scatter(self.y_test, test_pred, alpha=0.5)
        plt.plot([30, 100], [30, 100], 'r--')  # Perfect prediction line
        plt.title('Actual vs Predicted Performance Scores')
        plt.xlabel('Actual Score')
        plt.ylabel('Predicted Score')
        plt.grid(True)
        plt.savefig(os.path.join(plots_dir, "actual_vs_predicted.png"))
        plt.close()
        
        # Feature importance
        if hasattr(self.best_model, 'feature_importances_'):
            importances = self.best_model.feature_importances_
            indices = np.argsort(importances)[::-1]
            
            # Create DataFrame of feature importances
            self.feature_importance = pd.DataFrame({
                'Feature': [self.feature_columns[i] for i in indices],
                'Importance': [importances[i] for i in indices]
            })
            
            # Plot feature importance
            plt.figure(figsize=(12, 8))
            bars = plt.barh(
                range(len(indices)), 
                [importances[i] for i in indices], 
                align='center'
            )
            plt.yticks(range(len(indices)), 
                      [self.feature_columns[i] for i in indices])
            plt.title('Feature Importance')
            plt.xlabel('Relative Importance')
            plt.tight_layout()
            plt.savefig(os.path.join(plots_dir, "feature_importance.png"))
            plt.close()
            
            # Print feature importance
            print("\nFeature Importance:")
            for i, feature in enumerate(self.feature_importance['Feature']):
                print(f"  {i+1}. {feature}: {self.feature_importance['Importance'].iloc[i]:.2%}")
        
        # Residual analysis
        residuals = self.y_test - test_pred
        
        plt.figure(figsize=(10, 6))
        sns.histplot(residuals, kde=True)
        plt.title('Distribution of Prediction Errors')
        plt.xlabel('Prediction Error')
        plt.ylabel('Frequency')
        plt.savefig(os.path.join(plots_dir, "residual_distribution.png"))
        plt.close()
        
        # Residuals vs Predicted
        plt.figure(figsize=(10, 6))
        plt.scatter(test_pred, residuals, alpha=0.5)
        plt.axhline(y=0, color='r', linestyle='--')
        plt.title('Residuals vs Predicted Values')
        plt.xlabel('Predicted Values')
        plt.ylabel('Residuals')
        plt.grid(True)
        plt.savefig(os.path.join(plots_dir, "residuals_vs_predicted.png"))
        plt.close()
        
        print(f"\nEvaluation plots saved to: {plots_dir}")

    def save_model(self):
        """
        Save the best model with all necessary components for prediction
        """
        model_info = {
            'model': self.best_model,
            'scaler': self.scaler,
            'feature_columns': self.feature_columns,
            'model_name': self.best_model_name,
            'feature_importance': self.feature_importance if hasattr(self, 'feature_importance') else None,
            'metrics': self.test_metrics,
            'creation_date': self.creation_date,
            'created_by': self.user,
            'data_source': self.data_source
        }
        
        # Save model
        model_path = os.path.join(models_dir, "student_performance_model.pkl")
        with open(model_path, 'wb') as file:
            pickle.dump(model_info, file)
        
        print(f"\nModel saved to: {model_path}")
        
        # Create a simple model card text file with key information
        model_card = f"""
        # Student Performance Prediction Model
        
        ## Model Information
        - Model Type: {self.best_model_name}
        - Created by: {self.user}
        - Creation Date: {self.creation_date}
        - Data Source: {self.data_source}
        
        ## Performance Metrics
        - R² Score: {self.test_metrics['r2']:.4f}
        - Root Mean Squared Error: {self.test_metrics['rmse']:.2f}
        - Mean Absolute Error: {self.test_metrics['mae']:.2f}
        
        ## Features
        {', '.join(self.feature_columns)}
        
        ## Target
        {self.target_column}
        
        ## Usage
        This model predicts student performance scores based on study habits,
        previous grades, attendance, sleep patterns, and extracurricular activities.
        """
        
        # Save model card
        with open(os.path.join(models_dir, "model_card.txt"), 'w') as file:
            file.write(model_card)
        
        print("Model card created.")

    def predict_student_performance(self):
        """
        Interactive function to predict performance for a new student
        """
        print("\n" + "="*70)
        print("                  STUDENT PERFORMANCE PREDICTION")
        print("="*70)
        
        # Get user input for original features only
        base_features = {
            'Study Hours per Week': 'hours spent studying per week (5-40)',
            'Attendance Rate': 'percentage of classes attended (0-100)',
            'Previous Grades': 'average grades from previous term (0-100)',
            'Sleep Hours per Night': 'average hours of sleep per night (4-10)',
            'Extra Curricular Activities': 'participation in extracurricular activities (Yes/No)'
        }
        
        # Dictionary to store user inputs
        inputs = {}
        
        # Collect inputs for each feature
        for feature, description in base_features.items():
            if feature != 'Extra Curricular Activities':
                while True:
                    try:
                        value = float(input(f"Enter {feature} [{description}]: "))
                        inputs[feature] = value
                        break
                    except ValueError:
                        print("Please enter a numeric value.")
            else:
                while True:
                    value = input(f"Enter {feature} [{description}]: ").strip().capitalize()
                    if value in ['Yes', 'No']:
                        inputs[feature] = 1 if value == 'Yes' else 0
                        break
                    else:
                        print("Please enter 'Yes' or 'No'.")
        
        # Create input DataFrame with basic features
        input_df = pd.DataFrame([inputs])
        
        # Add engineered features
        input_df['Study_Efficiency'] = input_df['Previous Grades'] / input_df['Study Hours per Week']
        input_df['Study_Attendance'] = input_df['Study Hours per Week'] * input_df['Attendance Rate'] / 100
        input_df['Sleep_Optimality'] = 1 - abs(input_df['Sleep Hours per Night'] - 8) / 8
        
        # Ensure column order matches original feature set
        input_df = input_df[self.feature_columns]
        
        # Scale the input
        input_scaled = self.scaler.transform(input_df)
        
        # Generate prediction
        prediction = self.best_model.predict(input_scaled)[0]
        
        # Print prediction with confidence interval
        print("\n" + "-"*50)
        print(f"Predicted Performance Score: {prediction:.1f} / 100")
        print("-"*50)
        
        # Print interpretation
        print("\nInterpretation:")
        if prediction >= 90:
            print("  Outstanding performance - Excellent academic habits!")
        elif prediction >= 80:
            print("  Very good performance - Strong academic foundation.")
        elif prediction >= 70:
            print("  Good performance - Above average results.")
        elif prediction >= 60:
            print("  Satisfactory performance - Meeting basic expectations.")
        else:
            print("  Needs improvement - Consider academic support options.")
        
        # Generate personalized recommendations
        print("\nPersonalized Recommendations:")
        
        # Get the top 3 most important features
        if hasattr(self, 'feature_importance'):
            top_features = self.feature_importance['Feature'].iloc[:3].tolist()
        else:
            # Fallback if feature importance is not available
            top_features = ['Study Hours per Week', 'Previous Grades', 'Attendance Rate']
        
        # Study hours recommendation
        if 'Study Hours per Week' in top_features and inputs['Study Hours per Week'] < 20:
            print("  • Increase study time to at least 20 hours per week for better results.")
        
        # Attendance recommendation
        if 'Attendance Rate' in top_features and inputs['Attendance Rate'] < 85:
            print("  • Improve class attendance to at least 85% to better understand course material.")
        
        # Sleep recommendation
        if abs(inputs['Sleep Hours per Night'] - 8) > 1.5:
            if inputs['Sleep Hours per Night'] < 6.5:
                print("  • Consider getting more sleep (7-8 hours) for optimal cognitive function.")
            else:
                print("  • Aim for a more balanced sleep schedule (7-8 hours) for better performance.")
        
        # Extracurricular recommendation
        if inputs['Extra Curricular Activities'] == 0:
            print("  • Consider participating in extracurricular activities for a more well-rounded experience.")
        
        return prediction

def run_full_pipeline():
    """
    Run the complete model building pipeline
    """
    # Initialize predictor with synthetic data
    predictor = StudentPerformancePredictor(use_synthetic=True)
    
    # Run pipeline steps
    predictor.explore_data()
    predictor.prepare_data()
    predictor.train_models()
    predictor.evaluate_best_model()
    predictor.save_model()
    
    # Prediction interface
    while True:
        response = input("\nDo you want to predict a student's performance? (yes/no): ").strip().lower()
        if response in ['yes', 'y']:
            predictor.predict_student_performance()
        else:
            print("\nThank you for using the Student Performance Prediction System!")
            break

if __name__ == "__main__":
    run_full_pipeline()


                STUDENT PERFORMANCE PREDICTION SYSTEM
Created by: ShivamKashyap | Date: 2025-03-27 00:22:59

Generating synthetic student data...
Synthetic data saved to: C:\Users\Public\Placement\Student-Performace-Prediction\Data\synthetic_student_data.csv

Dataset shape: (500, 7)
Target variable: Performance Score

Sample data:
   Student_ID  Study Hours per Week  Attendance Rate  Previous Grades  \
0           1                  23.5             94.3             96.0   
1           2                  19.0            100.0             88.9   
2           3                  24.5             71.0             75.9   
3           4                  30.7             90.6             65.3   
4           5                  18.4             78.5             85.5   

   Sleep Hours per Night Extra Curricular Activities  Performance Score  
0                    8.2                          No               74.4  
1                    6.2                         Yes               75.6  
2    